In [ ]:
import looker_sdk
from looker_sdk import api_settings
from io import StringIO
import pandas as pd
from azure.storage.blob import BlobServiceClient
from io import BytesIO
import json
from datetime import datetime
from datetime import datetime, timedelta
import numpy as np 
import time

In [ ]:
# # ## chunk to connect to Looker API inAzure

# class KCRHA_Looker_API_Settings(api_settings.ApiSettings):
#     def __init__(self, *args, **kw_args):
#         self.my_var = kw_args.pop("my_var")
#         super().__init__(*args, **kw_args)

#     def read_config(self) -> api_settings.SettingsConfig:
#         config = super().read_config()
#         # See api_settings.SettingsConfig for required fields.
#         if self.my_var == "set":
#             config["base_url"] = TokenLibrary.getSecretWithLS('LS_AzureKeyVault','LOOKER-API-BASE-URL')
#             config["client_id"] = TokenLibrary.getSecretWithLS('LS_AzureKeyVault','LOOKER-API-CLIENT-ID')
#             config["client_secret"] = TokenLibrary.getSecretWithLS('LS_AzureKeyVault','LOOKER-API-CLIENT-SECRET')          
#         return config

# sdk = looker_sdk.init40(config_settings=KCRHA_Looker_API_Settings(my_var="set"))

# vars(vars(vars(sdk)["transport"])["settings"])["timeout"] = 300

In [ ]:
# Import Looker SDK connection from shared script
import sys
sys.path.insert(0, '..')
from looker_connection import sdk
from utils import write_table

In [ ]:
# Helper function to retry Looker queries with exponential backoff
def run_query_with_retry(sdk, body, result_format="csv", max_retries=5, base_delay=15):
    """
    Run a Looker inline query with retry logic for timeouts and server errors.
    Uses exponential backoff: 15, 30, 60, 120, 240 seconds between retries.
    """
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            return sdk.run_inline_query(result_format=result_format, body=body)
        except Exception as e:
            last_error = e
            error_str = str(e).lower()
            # Check for timeout, server errors, or dropped-connection errors that are worth retrying
            retryable = (
                "timeout" in error_str or "timed out" in error_str
                or "504" in str(e) or "503" in str(e) or "502" in str(e)
                or "response ended prematurely" in error_str
                or "connection reset" in error_str
                or "connection aborted" in error_str
                or "remote end closed connection" in error_str
            )
            if retryable:
                if attempt < max_retries:
                    wait_time = base_delay * (2 ** attempt)  # Exponential backoff: 15, 30, 60, 120, 240
                    print(f"Query failed ({e.__class__.__name__}). Retrying in {wait_time} seconds... (attempt {attempt + 1}/{max_retries})")
                    time.sleep(wait_time)
                    continue
            # For non-retryable errors, raise immediately
            raise
    raise last_error

In [ ]:
# Define your query

body = { 
"model": "wa_connection_model", 
"view": "client_model", 
"fields":[
    "client_model.unique_identifier",
    "client_model.personal_id",
    "client_model.added_date",
    #"client_model.id", removing field from data hub 
    "client_model.age",
    "client_model.age_tier",
    "static_demographics.other_tribe_specify",
    "static_demographics.federally_recognized_tribe",
    "static_demographics.gender_text",
    # Individual gender fields
    "static_demographics.gender_0_text",  # Woman
    "static_demographics.gender_1_text",  # Man
    "static_demographics.gender_2_text",  # Culturally Specific Identity
    "static_demographics.gender_3_text",  # Transgender
    "static_demographics.gender_4_text",  # Non-Binary
    "static_demographics.gender_5_text",  # Questioning
    "static_demographics.gender_6_text",  # A Different Identity
    "static_demographics.race_ethnicity_text",
    "static_demographics.veteran_text",
    # Race/ethnicity fields for consolidation
    "static_demographics.race_text",
    "static_demographics.ethnicity_text",
    "static_demographics.race_ethnicity_1_text",  # American Indian, Alaska Native, or Indigenous
    "static_demographics.race_ethnicity_2_text",  # Asian or Asian American
    "static_demographics.race_ethnicity_3_text",  # Black, African American, or African
    "static_demographics.race_ethnicity_4_text",  # Native Hawaiian or Pacific Islander
    "static_demographics.race_ethnicity_5_text",  # White
    "static_demographics.race_ethnicity_6_text",  # Hispanic/Latina/o
    "static_demographics.race_ethnicity_7_text"   # Middle Eastern or North African
    ], 
#FILTER to remove deleted records 
"filters": {"client_model.deleted" : "No"},
"limit": -1
            }  

################ 
# Get Response 
################ 

RESULT_FORMAT = "csv" 

# Run the inline query with retry logic
result = run_query_with_retry(sdk, body, result_format=RESULT_FORMAT)

# Convert to dataframe.
df0 = pd.read_csv(StringIO(result))

In [ ]:
# Consolidate race/ethnicity columns: update boolean fields based on text fields, then drop text fields

race_col = "Clients Race"
ethnicity_col = "Clients Ethnicity"

# Mapping of text patterns to their corresponding boolean columns
race_bool_map = {
    "American Indian": "Clients American Indian, Alaska Native, or Indigenous",
    "Alaska Native": "Clients American Indian, Alaska Native, or Indigenous",
    "Indigenous": "Clients American Indian, Alaska Native, or Indigenous",
    "Asian": "Clients Asian or Asian American",
    "Black": "Clients Black, African American, or African",
    "African American": "Clients Black, African American, or African",
    "Native Hawaiian": "Clients Native Hawaiian or Pacific Islander",
    "Pacific Islander": "Clients Native Hawaiian or Pacific Islander",
    "White": "Clients White",
    "Middle Eastern": "Clients Middle Eastern or North African",
}

# Update Hispanic boolean where Ethnicity says Hispanic but boolean is No
hispanic_bool_col = "Clients Hispanic/Latina/o"
hispanic_mask = (
    df0[ethnicity_col].str.contains("Hispanic/Latin", case=False, na=False, regex=False) & 
    ~df0[ethnicity_col].str.contains("Non-Hispanic", case=False, na=False, regex=False) &
    (df0[hispanic_bool_col] == "No")
)
df0.loc[hispanic_mask, hispanic_bool_col] = "Yes"
print(f"Updated {hispanic_mask.sum()} rows: Hispanic ethnicity -> Hispanic boolean = Yes")

# Update race booleans where Clients Race contains info not reflected in boolean
for pattern, bool_col in race_bool_map.items():
    race_mask = (
        df0[race_col].str.contains(pattern, case=False, na=False, regex=False) &
        (df0[bool_col] == "No")
    )
    if race_mask.sum() > 0:
        df0.loc[race_mask, bool_col] = "Yes"
        print(f"Updated {race_mask.sum()} rows: Race contains '{pattern}' -> {bool_col} = Yes")

# Drop the text columns used for consolidation
df0.drop(columns=[race_col, ethnicity_col], inplace=True)
print(f"Dropped columns: {race_col}, {ethnicity_col}")

In [ ]:
columnHeaders = {
    "Clients Unique Identifier":                          "ClientUniqueIdentifier",
    "Clients Personal ID":                                "PersonalID",
    "Clients Date Created Date":                          "ClientProfileCreatedDate",
    "Clients Client ID":                                  "ClientID",
    "Clients Current Age":                                "CurrentAge",
    "Clients Age Tier":                                   "AgeTier",
    "Clients Other Tribe Specify":                        "OtherTribe",
    "Clients Tribes List":                                "FederallyRecognizedTribe",
    "Clients Gender":                                     "Gender",
    "Clients Race and Ethnicity":                         "RaceAndEthnicity",
    "Clients Veteran Status":                             "VeteranStatus",
    # Race/ethnicity boolean columns
    "Clients American Indian, Alaska Native, or Indigenous": "RaceAndEthnicity_AIAN",
    "Clients Asian or Asian American":                    "RaceAndEthnicity_Asian",
    "Clients Black, African American, or African":        "RaceAndEthnicity_Black",
    "Clients Native Hawaiian or Pacific Islander":        "RaceAndEthnicity_NHPI",
    "Clients White":                                      "RaceAndEthnicity_WC",
    "Clients Hispanic/Latina/o":                          "RaceAndEthnicity_HL",
    "Clients Middle Eastern or North African":            "RaceAndEthnicity_MENA",
    # Gender boolean columns
    "Clients Woman (Girl, if child)":                     "Gender_Woman",
    "Clients Man (Boy, if Child)":                        "Gender_Man",
    "Clients Culturally Specific Identity (e.g., Two-Spirit)": "Gender_CSI",
    "Clients Transgender":                                "Gender_Trans",
    "Clients Non-Binary":                                 "Gender_NB",
    "Clients Questioning":                                "Gender_Q",
    "Clients Different Identity":                         "Gender_DI",
}
df0.rename(columns=columnHeaders, inplace=True)

In [ ]:
# Drop client records with no PersonalID — HMIS 5.08 should be populated on
# every client, so a blank one means an incomplete/orphaned profile in Clarity.
# These can't be joined to anything else in this notebook anyway, and pandas'
# drop_duplicates() treats all NaN PersonalIDs as equal, which would otherwise
# silently collapse several distinct people missing a PersonalID into one row.
n_no_pid = df0["PersonalID"].isna().sum()
if n_no_pid > 0:
    print(f"⚠ Dropping {n_no_pid:,} client_model row(s) with no PersonalID (data quality issue upstream)")
    df0 = df0[df0["PersonalID"].notna()].reset_index(drop=True)

df0["ClientProfileCreatedDate"] = pd.to_datetime(df0["ClientProfileCreatedDate"], errors="coerce")

# ClientUniqueIdentifier: per code review feedback, keep this column but
# take it from the EARLIEST-created client record per PersonalID (the
# identifier assigned when the person's profile was first created), rather
# than whichever record wins the demographic collapse below. This is a
# placeholder rule — no ProgramType-based hierarchy has been defined yet.
earliest_client_id = (
    df0.sort_values("ClientProfileCreatedDate", ascending=True)
    .drop_duplicates(subset="PersonalID", keep="first")
    [["PersonalID", "ClientUniqueIdentifier"]]
)
df0 = df0.drop(columns="ClientUniqueIdentifier")

# Collapse to one row per PersonalID — one PersonalID can have multiple client
# records (multiple ClientUniqueIdentifiers) in client_model, which fanned out
# downstream joins. Before collapsing, report any PersonalID whose client
# records actually disagree on a demographic value (a real data-quality issue
# worth reviewing), then keep the row from the most recently created client
# profile.
_dedup_check_cols = [c for c in df0.columns if c not in ("PersonalID", "ClientProfileCreatedDate")]
_conflicting_ids = df0.groupby("PersonalID")[_dedup_check_cols].nunique(dropna=True).gt(1).any(axis=1)
_conflicting_ids = _conflicting_ids[_conflicting_ids].index

if len(_conflicting_ids) > 0:
    print(f"⚠ {len(_conflicting_ids):,} PersonalIDs have conflicting demographic values across multiple client records:")
    display(df0[df0["PersonalID"].isin(_conflicting_ids)].sort_values("PersonalID"))
else:
    print("No conflicting demographic values found across duplicate PersonalIDs.")

n_before = len(df0)
df0 = (
    df0.sort_values("ClientProfileCreatedDate", ascending=False)
    .drop_duplicates(subset="PersonalID", keep="first")
    .drop(columns="ClientProfileCreatedDate")
    .merge(earliest_client_id, on="PersonalID", how="left")
    .reset_index(drop=True)
)
print(f"\nCollapsed {n_before:,} client records -> {len(df0):,} rows, one per PersonalID")

In [ ]:
# ---------------------------------------------------------------------------
# Checked FederallyRecognizedTribe against static_demographics, entry_screen,
# followup_screen, status_update_screen, last_screen, and client_assessments
# too. Only dq_client_demographics found meaningful net-new coverage
# ---------------------------------------------------------------------------
result_tribe_dq = run_query_with_retry(sdk, {
    "model": "wa_connection_model",
    "view": "data_quality",
    "fields": [
        "data_quality.personal_id",
        "dq_client_demographics.federally_recognized_tribe",
    ],
    "limit": "-1",
})
tribe_dq = pd.read_csv(StringIO(result_tribe_dq), low_memory=False)
tribe_dq.columns = ["PersonalID", "FederallyRecognizedTribe"]
tribe_dq["source"] = "dq_client_demographics"
print(f"✔ dq_client_demographics: {len(tribe_dq):,} rows")


In [ ]:
# ---------------------------------------------------------------------------
# Backfill FederallyRecognizedTribe from dq_client_demographics
# ---------------------------------------------------------------------------
TRIBE_DQ_TEXT_LOWER = {"client doesn't know", "client prefers not to answer", "data not collected"}

def _tribe_normalize_text(series):
    return series.astype(str).str.lower().str.replace("’", "'", regex=False)

def tribe_valid_mask(series):
    return series.notna() & ~_tribe_normalize_text(series).isin(TRIBE_DQ_TEXT_LOWER)

dq_fill = (
    tribe_dq[tribe_valid_mask(tribe_dq["FederallyRecognizedTribe"])]
    .drop_duplicates(subset="PersonalID", keep="first")
    .set_index("PersonalID")["FederallyRecognizedTribe"]
)

needs_fill = ~tribe_valid_mask(df0["FederallyRecognizedTribe"])
fillable = df0["PersonalID"].isin(dq_fill.index) & needs_fill

n_filled = fillable.sum()
df0.loc[fillable, "FederallyRecognizedTribe"] = df0.loc[fillable, "PersonalID"].map(dq_fill)

print(f"Backfilled FederallyRecognizedTribe for {n_filled:,} clients from dq_client_demographics")
print(f"{tribe_valid_mask(df0['FederallyRecognizedTribe']).sum():,} / {len(df0):,} clients now have a valid value")


In [ ]:
# Build RaceAndEthnicity and RaceAndEthnicityExpanded from binary fields only.
#
# RaceAndEthnicity — summary rollup:
#   White + Hispanic only  → Hispanic/Latina/o
#   Any other 2+ flags     → Multiracial
#   Exactly 1 flag         → that single label
#   0 flags                → Unknown
#
# RaceAndEthnicityExpanded — granular:
#   White + Hispanic only  → Hispanic/Latina/o
#   Exactly 1 flag         → that single label
#   Exactly 2 flags        → "Label A & Label B"
#   3+ flags               → Multiracial — with/without Hispanic/Latina/o
#   0 flags                → Unknown

RACE_COLS = [
    ('RaceAndEthnicity_AIAN', 'American Indian, Alaska Native, or Indigenous'),
    ('RaceAndEthnicity_Asian', 'Asian or Asian American'),
    ('RaceAndEthnicity_Black', 'Black, African American, or African'),
    ('RaceAndEthnicity_NHPI', 'Native Hawaiian or Pacific Islander'),
    ('RaceAndEthnicity_WC', 'White'),
    ('RaceAndEthnicity_HL', 'Hispanic/Latina/o'),
    ('RaceAndEthnicity_MENA', 'Middle Eastern or North African'),
]

# Boolean matrix: True where each flag is 'Yes'
flags = pd.concat(
    {label: df0[col] == 'Yes' for col, label in RACE_COLS},
    axis=1
)
yes_count = flags.sum(axis=1)

# Shared masks
is_single        = yes_count == 1
is_white_hl_only = (yes_count == 2) & flags['White'] & flags['Hispanic/Latina/o']
is_multi         = (yes_count >= 2) & ~is_white_hl_only
is_multi_hl      = is_multi & flags['Hispanic/Latina/o']
is_multi_no_hl   = is_multi & ~flags['Hispanic/Latina/o']

single_label = flags.idxmax(axis=1).where(is_single)

# RaceAndEthnicity — summary rollup (no with/without distinction)
race_result = pd.Series('Unknown', index=df0.index)
race_result[is_single]       = single_label[is_single]
race_result[is_white_hl_only] = 'Hispanic/Latina/o'
race_result[is_multi]        = 'Multiracial'
df0['RaceAndEthnicity'] = race_result

# RaceAndEthnicityExpanded — granular (2-flag combos spelled out; 3+ collapsed to Multiracial with/without)
def two_race_label(row):
    return ' & '.join(c for c in flags.columns if row[c])

two_label = flags[yes_count == 2].apply(two_race_label, axis=1)

expanded_result = pd.Series('Unknown', index=df0.index)
expanded_result[is_single]        = single_label[is_single]
expanded_result[yes_count == 2]   = two_label
expanded_result[is_white_hl_only] = 'Hispanic/Latina/o'   # override "White & Hispanic/Latina/o"
expanded_result[is_multi_hl & (yes_count >= 3)]    = 'Multiracial — with Hispanic/Latina/o'
expanded_result[is_multi_no_hl & (yes_count >= 3)] = 'Multiracial — without Hispanic/Latina/o'
df0['RaceAndEthnicityExpanded'] = expanded_result

print(df0['RaceAndEthnicity'].value_counts())
print()
print(df0['RaceAndEthnicityExpanded'].value_counts())

In [ ]:
# Build GenderExpanded and override Gender for multiple selections or DK/PNTA/DNC
# Valid gender codes 0-6; 8/9/99 (DK/PNTA/DNC) all map to Unknown

GENDER_COLS = [
    ('Gender_Woman', 'Woman'),
    ('Gender_Man', 'Man'),
    ('Gender_CSI', 'Culturally Specific Identity'),
    ('Gender_Trans', 'Transgender'),
    ('Gender_NB', 'Non-Binary'),
    ('Gender_Q', 'Questioning'),
    ('Gender_DI', 'A Different Identity'),
]

# Boolean matrix: True where each flag is 'Yes'
gender_flags = pd.concat(
    {label: df0[col] == 'Yes' for col, label in GENDER_COLS},
    axis=1
)
gender_yes_count = gender_flags.sum(axis=1)

# n==1: single label
gender_single = gender_flags.idxmax(axis=1).where(gender_yes_count == 1)

# n==2: concatenate the two selected labels in column order
def two_gender_label(row):
    return ' & '.join(c for c in gender_flags.columns if row[c])

gender_two = gender_flags[gender_yes_count == 2].apply(two_gender_label, axis=1)

# Assemble GenderExpanded: start with 3+ default, overwrite lower-n cases
gender_result = pd.Series('Three or more genders selected', index=df0.index)
gender_result[gender_yes_count == 2] = gender_two
gender_result[gender_yes_count == 1] = gender_single[gender_yes_count == 1]
gender_result[gender_yes_count == 0] = 'Unknown'

df0['GenderExpanded'] = gender_result

# Override Gender text for multi-select and DK/PNTA/DNC
df0.loc[gender_yes_count >= 2, 'Gender'] = 'Multiple genders selected'
df0.loc[gender_yes_count == 0, 'Gender'] = 'Unknown'

print(df0['GenderExpanded'].value_counts())
print(f"\nOverrode {(gender_yes_count >= 2).sum()} rows: Gender -> 'Multiple genders selected'")
print(f"Overrode {(gender_yes_count == 0).sum()} rows: Gender -> 'Unknown'")

# Reorder: place expanded columns immediately after their parent
cols = list(df0.columns)
for col, after in [('GenderExpanded', 'Gender'), ('RaceAndEthnicityExpanded', 'RaceAndEthnicity')]:
    cols.remove(col)
    cols.insert(cols.index(after) + 1, col)
df0 = df0[cols]

In [ ]:
#add timestamp
today_date = datetime.today().date()
df0['DataAsOfDate'] = today_date


In [ ]:
# Correct ages 110 and older to NaN and "Undefined"
mask = pd.to_numeric(df0["CurrentAge"], errors="coerce") >= 110
df0.loc[mask, ["CurrentAge", "AgeTier"]] = [np.nan, "Undefined"]

# Convert CurrentAge to nullable integer (preserves NaN without trailing .0)
df0["CurrentAge"] = pd.to_numeric(df0["CurrentAge"], errors="coerce").astype("Int64")

In [ ]:
import pgeocode

UNINFORMATIVE = {
    "Washington State (outside of King County)",
    "Outside of Washington State",
    "Data not collected",
    "Client prefers not to answer",
    "Client doesn't know",
}

zip_fields = {"client_addresses_geolocations.zipcode"}

# Precedence across candidate sources: prior residence is more meaningful than
# last permanent residence (which can reflect somewhere a client lived long
# before becoming homeless), so prior should win even when its record is
# older. Address city and zip-derived city are lower-confidence fallbacks.
# This ranks ahead of recency; recency still breaks ties within a field.
FIELD_PRECEDENCE = {
    "entry_custom.c_KC_jurisdiction_prior": 0,
    "entry_custom.c_kc_jurisdiction_last_permanent": 1,
    "client_addresses_geolocations.city": 2,
    "client_addresses_geolocations.zipcode": 3,
}

_lpl_batches = [
    {
        "fields": [
            "clients.personal_id",
            "enrollments.start_date",
            "entry_custom.c_kc_jurisdiction_last_permanent",
            "entry_custom.c_KC_jurisdiction_prior",
        ],
        "filters": {
            "enrollments.date_filter": "NULL",
        },
        "date_col": "enrollments.start_date",
    },
    {
        "fields": [
            "clients.personal_id",
            "client_addresses.added_date",
            "client_addresses_geolocations.city",
            "client_addresses_geolocations.zipcode",
        ],
        "filters": {
            "enrollments.date_filter": "NULL",
            "client_addresses.is_latest_address": "Yes",
        },
        "date_col": "client_addresses.added_date",
    },
]

lpl_results = {}
for batch in _lpl_batches:
    candidate_fields = batch["fields"][2:]
    try:
        raw = run_query_with_retry(
            sdk,
            looker_sdk.models40.WriteQuery(
                model="wa_connection_model",
                view="base",
                fields=batch["fields"],
                filters=batch["filters"],
                limit="-1",
            ),
        )
        df_batch = pd.read_csv(StringIO(raw))
        assert len(df_batch.columns) == len(batch["fields"]), (
            f"Expected {len(batch['fields'])} columns, got {len(df_batch.columns)}: {list(df_batch.columns)}"
        )
        df_batch.columns = ["PersonalID", "date"] + candidate_fields
        df_batch["date"] = pd.to_datetime(df_batch["date"], errors="coerce")
        long = df_batch.melt(
            id_vars=["PersonalID", "date"], value_vars=candidate_fields,
            var_name="field", value_name="value",
        )
        for field in candidate_fields:
            lpl_results[field] = (
                long[long["field"] == field]
                .dropna(subset=["value"])
                .drop(columns="field")
                .sort_values("date", ascending=False)
                .drop_duplicates(subset=["PersonalID"])
                .reset_index(drop=True)
            )
    except Exception as e:
        print(f"Last place lived batch failed ({batch['date_col']}): {e}")

stacked = pd.concat(
    [df.assign(field=field) for field, df in lpl_results.items() if len(df)],
    ignore_index=True,
)
stacked = (
    stacked.dropna(subset=["value", "date"])
    .loc[lambda x: x["value"].astype(str).str.strip() != ""]
    .sort_values("date", ascending=False)
    .reset_index(drop=True)
)

# Light normalization: strip "City of" prefix, trim, and collapse internal
# whitespace. Case-insensitive UNINFORMATIVE check below (rather than
# title-casing before comparing) so it still matches values regardless of
# how they were capitalized on entry.
#
# NOTE: fuzzy-matching client_addresses_geolocations.city (the free-text
# fallback, ~9% of records) against a valid-city list was tried and reverted
# — even at a conservative threshold (85), it silently folded real non-KC
# places into King County cities (Moses Lake, Soap Lake, Bend OR, Lopez
# Island, Rock Island, Ocean Park, North Hollywood all matched onto KC
# cities purely on shared words like "Lake"/"Island"/"North"). That's a worse
# outcome than leaving the free text unstandardized, so this field gets only
# the same light normalization as everything else — no canonical-list
# matching without a real WA gazetteer and per-value sign-off.
stacked["value"] = (
    stacked["value"].astype(str)
    .str.replace(r"(?i)^city of\s+", "", regex=True)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)
stacked["is_uninformative"] = stacked["value"].str.lower().isin({u.lower() for u in UNINFORMATIVE}).astype(int)
stacked["precedence"] = stacked["field"].map(FIELD_PRECEDENCE)

most_recent_city = (
    stacked.sort_values(["is_uninformative", "precedence", "date"], ascending=[True, True, False])
    .drop_duplicates(subset=["PersonalID"])
    .drop(columns=["is_uninformative", "precedence"])
    .reset_index(drop=True)
)

# Map zip code winners → city names
nomi = pgeocode.Nominatim("us")

def _clean_zip(s):
    try:
        return str(int(float(s))).zfill(5)
    except (ValueError, TypeError):
        return None

zip_mask = most_recent_city["field"].isin(zip_fields)
if zip_mask.any():
    raw_zips = most_recent_city.loc[zip_mask, "value"].dropna().unique()
    clean_zips = [z for z in (_clean_zip(z) for z in raw_zips) if z]
    if clean_zips:
        lookup = (
            nomi.query_postal_code(clean_zips)[["postal_code", "place_name"]]
            .rename(columns={"postal_code": "zip", "place_name": "city"})
            .dropna(subset=["city"])
        )
        lookup["zip"] = lookup["zip"].astype(str).str.zfill(5)
        zip_to_city = lookup.set_index("zip")["city"].to_dict()
        most_recent_city.loc[zip_mask, "value"] = (
            most_recent_city.loc[zip_mask, "value"].apply(_clean_zip).map(zip_to_city)
        )

# Final display-casing pass, applied uniformly across all four sources
# (free-text jurisdiction/address fields + pgeocode place names) so casing
# is consistent regardless of which source won.
most_recent_city["value"] = most_recent_city["value"].astype(str).str.strip().str.title()

# City → LastKnownSubregion mapping (King County). The c_KC_jurisdiction_prior /
# c_kc_jurisdiction_last_permanent dropdown is a controlled vocabulary (~55
# values covering ~91% of records) with its own exact spellings, which don't
# always match the original subregion-mapping list (e.g. "Beaux Arts" not
# "Beaux Arts Village", "Sea Tac" not "SeaTac", "Vashon/Maury Island" not
# "Vashon") — both spellings are kept here so zip/address-city-derived values
# still match too. "(Unincorporated)" areas are left unmapped
# (LastKnownSubregion = NaN). Enumclaw and Lake Forest Park (real KC cities
# missing from the original list) map to Southeast and North King County
# respectively, per code review feedback.
CITY_SUBREGION = pd.DataFrame([
    {"City": "Algona", "LastKnownSubregion": "South King County"},
    {"City": "Auburn", "LastKnownSubregion": "South King County"},
    {"City": "Beaux Arts", "LastKnownSubregion": "East King County"},
    {"City": "Beaux Arts Village", "LastKnownSubregion": "East King County"},
    {"City": "Bellevue", "LastKnownSubregion": "East King County"},
    {"City": "Black Diamond", "LastKnownSubregion": "Southeast King County"},
    {"City": "Bothell", "LastKnownSubregion": "North King County"},
    {"City": "Burien", "LastKnownSubregion": "South King County"},
    {"City": "Carnation", "LastKnownSubregion": "Snoqualmie Valley"},
    {"City": "Clyde Hill", "LastKnownSubregion": "East King County"},
    {"City": "Covington", "LastKnownSubregion": "Southeast King County"},
    {"City": "Des Moines", "LastKnownSubregion": "South King County"},
    {"City": "Duvall", "LastKnownSubregion": "Snoqualmie Valley"},
    {"City": "Enumclaw", "LastKnownSubregion": "Southeast King County"},
    {"City": "Federal Way", "LastKnownSubregion": "South King County"},
    {"City": "Hunts Point", "LastKnownSubregion": "East King County"},
    {"City": "Issaquah", "LastKnownSubregion": "East King County"},
    {"City": "Kenmore", "LastKnownSubregion": "North King County"},
    {"City": "Kent", "LastKnownSubregion": "South King County"},
    {"City": "Kirkland", "LastKnownSubregion": "East King County"},
    {"City": "Lake Forest Park", "LastKnownSubregion": "North King County"},
    {"City": "Maple Valley", "LastKnownSubregion": "Southeast King County"},
    {"City": "Medina", "LastKnownSubregion": "East King County"},
    {"City": "Mercer Island", "LastKnownSubregion": "Seattle Metro"},
    {"City": "Milton", "LastKnownSubregion": "South King County"},
    {"City": "Newcastle", "LastKnownSubregion": "East King County"},
    {"City": "Normandy Park", "LastKnownSubregion": "South King County"},
    {"City": "North Bend", "LastKnownSubregion": "Snoqualmie Valley"},
    {"City": "Pacific", "LastKnownSubregion": "South King County"},
    {"City": "Redmond", "LastKnownSubregion": "East King County"},
    {"City": "Renton", "LastKnownSubregion": "South King County"},
    {"City": "Sammamish", "LastKnownSubregion": "East King County"},
    {"City": "Sea Tac", "LastKnownSubregion": "South King County"},
    {"City": "Seatac", "LastKnownSubregion": "South King County"},
    {"City": "Seattle", "LastKnownSubregion": "Seattle Metro"},
    {"City": "Shoreline", "LastKnownSubregion": "North King County"},
    {"City": "Skykomish", "LastKnownSubregion": "Snoqualmie Valley"},
    {"City": "Snoqualmie", "LastKnownSubregion": "Snoqualmie Valley"},
    {"City": "Tukwila", "LastKnownSubregion": "South King County"},
    {"City": "Vashon", "LastKnownSubregion": "Seattle Metro"},
    {"City": "Vashon/Maury Island", "LastKnownSubregion": "Seattle Metro"},
    {"City": "Woodinville", "LastKnownSubregion": "North King County"},
    {"City": "Yarrow Point", "LastKnownSubregion": "East King County"},
])

most_recent_city = (
    most_recent_city[["PersonalID", "value"]]
    .rename(columns={"value": "LastKnownCity"})
    .merge(CITY_SUBREGION, left_on="LastKnownCity", right_on="City", how="left")
    .drop(columns="City")
)

df0 = df0.merge(most_recent_city, on="PersonalID", how="left")

In [ ]:
# VeteranStatus: map anything that isn't Yes/No to Unknown; null out for minors (age < 18)
valid_vet = {'Yes', 'No'}
df0.loc[df0['VeteranStatus'].notna() & ~df0['VeteranStatus'].isin(valid_vet), 'VeteranStatus'] = 'Unknown'
df0.loc[df0['CurrentAge'] < 18, 'VeteranStatus'] = pd.NA

print(df0['VeteranStatus'].value_counts(dropna=False))

In [ ]:
# ---------------------------------------------------------------------------
# Pull disability, income, DV/orientation/translation/foster-care/justice, and
# living-situation text fields from all screen views — queries run in parallel
# Fields:
#   disabling_condition  (3.08) — 0 / 1
#   health_mental        (4.09) — 0 / 1
#   health_substance_abuse (4.10) — 0=None / 1=Alcohol / 2=Drug / 3=Both
#   income_cash          — sum of cash income from any source (total monthly)
#   health_dv (4.11.2), health_dv_fleeing (4.11.2B, conditional on health_dv=Yes),
#   rhy_sexual_orientation, translation_assistance_needed,
#   previous_foster_care, rhy_former_justice — see field-availability notes below
#   prior_residence_text, exit_destination_text — scanned in a later cell for
#     institutional living situations (foster care / justice / behavioral health)
#
# Field availability differs by screen (confirmed against the Looker Data
# Dictionary), so each source declares its own "fields" list rather than a
# single field list applied uniformly to every source.
# ---------------------------------------------------------------------------
from concurrent.futures import ThreadPoolExecutor, as_completed

# Superset of every optional field any source might request. Any field a given
# source doesn't ask for is backfilled with NaN after the query returns, which
# generalizes the income_cash-only backfill this notebook used previously.
MASTER_FIELDS = [
    "disabling_condition", "health_mental", "health_substance_abuse", "income_cash",
    "health_dv", "health_dv_fleeing", "rhy_sexual_orientation", "translation_assistance_needed",
    "previous_foster_care", "rhy_former_justice",
    "prior_residence_text", "exit_destination_text",
]

_disability = ["disabling_condition", "health_mental", "health_substance_abuse"]

sources = {
    "entry_screen": {
        "explore":  "client_model",
        "view":     "entry_screen",
        "id_field": "client_model.personal_id",
        "filters":  {"client_model.deleted": "No"},
        "fields": _disability + [
            "income_cash", "health_dv", "health_dv_fleeing", "rhy_sexual_orientation",
            "translation_assistance_needed", "previous_foster_care", "rhy_former_justice",
            "prior_residence_text",
        ],
    },
    "followup_screen": {
        "explore":  "client_model",
        "view":     "followup_screen",
        "id_field": "client_model.personal_id",
        "filters":  {"client_model.deleted": "No"},
        "fields": _disability + ["income_cash", "prior_residence_text"],
    },
    "status_update_screen": {
        "explore":  "client_model",
        "view":     "status_update_screen",
        "id_field": "client_model.personal_id",
        "filters":  {"client_model.deleted": "No"},
        "fields": _disability + [
            "income_cash", "health_dv", "health_dv_fleeing", "prior_residence_text",
        ],
    },
    "last_screen": {
        "explore":  "client_model",
        "view":     "last_screen",
        "id_field": "client_model.personal_id",
        "filters":  {"client_model.deleted": "No"},
        "fields": _disability + [
            "income_cash", "health_dv", "health_dv_fleeing", "prior_residence_text", "exit_destination_text",
        ],
    },
    "dq_client_program_demographics": {
        "explore":  "data_quality",
        "view":     "dq_client_program_demographics",
        "id_field": "data_quality.personal_id",
        "fields": _disability + [
            "health_dv", "health_dv_fleeing", "rhy_sexual_orientation", "previous_foster_care", "rhy_former_justice",
            "prior_residence_text", "exit_destination_text",
        ],  # income_cash not available in this view
    },
}

def fetch_source(source_name, cfg):
    v = cfg["view"]
    requested = cfg["fields"]

    body = {
        "model":   "wa_connection_model",
        "view":    cfg["explore"],
        "fields":  [
            cfg["id_field"],
            f"{v}.program_date",
            *[f"{v}.{f}" for f in requested],
        ],
        "filters": cfg.get("filters", {}),
        "limit":   "-1",
    }
    result = run_query_with_retry(sdk, body, result_format="csv")
    df = pd.read_csv(StringIO(result))
    df.columns = ["PersonalID", "date", *requested]
    for f in MASTER_FIELDS:
        if f not in df.columns:
            df[f] = pd.NA
    df["source"] = source_name
    return source_name, df

raw    = {}
errors = {}

with ThreadPoolExecutor(max_workers=6) as executor:
    futures = {executor.submit(fetch_source, name, cfg): name for name, cfg in sources.items()}
    for future in as_completed(futures):
        source_name = futures[future]
        try:
            name, df = future.result()
            raw[name] = df
            print(f"✔ {name}: {len(df):,} rows")
        except Exception as e:
            errors[source_name] = str(e)
            print(f"✖ {source_name}: {e}")

In [ ]:
# ---------------------------------------------------------------------------
# Pull DV/orientation/foster-care/justice fields and prior-residence text from
# client_assessments, plus current living situation, both keyed on PersonalID
# directly (same as every other pull in this notebook — no ClientUniqueIdentifier
# crosswalk needed). "enrollments.date_filter": "NULL" is required on the base
# explore or it silently limits to the last 90 days.
# ---------------------------------------------------------------------------

assess_fields = [
    "clients.personal_id",
    "client_assessments.assessment_date",
    "client_assessments.health_dv",
    "client_assessments.health_dv_fleeing",
    "client_assessments.rhy_sexual_orientation",
    "client_assessments.previous_foster_care",
    "client_assessments.rhy_former_justice",
    "client_assessments.prior_residence_text",
]

result_assess = run_query_with_retry(sdk, {
    "model": "wa_connection_model",
    "view": "base",
    "fields": assess_fields,
    "filters": {"enrollments.date_filter": "NULL"},
    "limit": "-1",
})
assess_raw = pd.read_csv(StringIO(result_assess), low_memory=False)

if len(assess_raw.columns) != len(assess_fields):
    raise ValueError(f"client_assessments column count mismatch: expected {len(assess_fields)}, got {len(assess_raw.columns)}")
assess_raw.columns = [
    "PersonalID", "AssessmentDate", "health_dv", "health_dv_fleeing", "rhy_sexual_orientation",
    "previous_foster_care", "rhy_former_justice", "prior_residence_text",
]
assess_raw["AssessmentDate"] = pd.to_datetime(assess_raw["AssessmentDate"], errors="coerce")
print(f"{len(assess_raw):,} client_assessments rows for {assess_raw['PersonalID'].nunique():,} clients")

# Current Living Situation — a different explore shape (no program_date; uses
# information_date instead), so it needs its own dedicated query rather than
# fitting into the fetch_source/sources pattern above.
cls_fields = [
    "clients.personal_id",
    "current_living_situation.current_living_situation",
    "current_living_situation.information_date_date",
]

result_cls = run_query_with_retry(sdk, {
    "model": "wa_connection_model",
    "view": "base",
    "fields": cls_fields,
    "filters": {
        "enrollments.date_filter": "NULL",
        "current_living_situation.information_date_date": "after 2017-01-01",
    },
    "limit": "-1",
})
cls_raw = pd.read_csv(StringIO(result_cls), low_memory=False)

if len(cls_raw.columns) != len(cls_fields):
    raise ValueError(f"current_living_situation column count mismatch: expected {len(cls_fields)}, got {len(cls_raw.columns)}")
cls_raw.columns = ["PersonalID", "CurrentLivingSituation", "InformationDate"]
cls_raw["InformationDate"] = pd.to_datetime(cls_raw["InformationDate"], errors="coerce")
print(f"{len(cls_raw):,} current_living_situation rows for {cls_raw['PersonalID'].nunique():,} clients")

In [ ]:
# Pull HMIS education fields from base (entry/last screen are enrollment-level, no fan-out risk)
# filter_expression restricts to rows where at least one education field is non-null,
# reducing the result set before transfer
edu_fields = [
    "clients.personal_id",
    "enrollments.start_date",
    "enrollments.end_date",
    "entry_screen.rhy_education_level",
    "entry_screen.rhy_school_status",
    "entry_screen.youth_education_current_status",
    "entry_screen.youth_education_recent_status",
    "entry_screen.youth_education_enrollment",
    "last_screen.rhy_education_level",
    "last_screen.rhy_school_status",
    "last_screen.youth_education_current_status",
    "last_screen.youth_education_recent_status",
    "last_screen.youth_education_enrollment",
]

edu_filter_expr = (
    "NOT ("
    "is_null(${entry_screen.rhy_education_level}) AND "
    "is_null(${entry_screen.rhy_school_status}) AND "
    "is_null(${entry_screen.youth_education_current_status}) AND "
    "is_null(${entry_screen.youth_education_recent_status}) AND "
    "is_null(${entry_screen.youth_education_enrollment}) AND "
    "is_null(${last_screen.rhy_education_level}) AND "
    "is_null(${last_screen.rhy_school_status}) AND "
    "is_null(${last_screen.youth_education_current_status}) AND "
    "is_null(${last_screen.youth_education_recent_status}) AND "
    "is_null(${last_screen.youth_education_enrollment})"
    ")"
)

result = run_query_with_retry(sdk, {
    "model": "wa_connection_model",
    "view": "base",
    "fields": edu_fields,
    "filters": {"enrollments.date_filter": "NULL"},
    "filter_expression": edu_filter_expr,
    "limit": "-1",
})
edu_raw = pd.read_csv(StringIO(result), low_memory=False)

if len(edu_raw.columns) != len(edu_fields):
    raise ValueError(f"Column count mismatch: expected {len(edu_fields)}, got {len(edu_raw.columns)}")
edu_raw.columns = edu_fields

edu_raw["enrollments.start_date"] = pd.to_datetime(edu_raw["enrollments.start_date"], errors="coerce")
edu_raw["enrollments.end_date"] = pd.to_datetime(edu_raw["enrollments.end_date"], errors="coerce")
print(f"{len(edu_raw):,} enrollment rows for {edu_raw['clients.personal_id'].nunique():,} clients")

# DQ note: last_screen rows that have an education value but no exit date
def has_edu_value(series):
    return series.notna() & ~series.astype(str).str.strip().isin(["", "99", "99.0", "nan"])

last_screen_cols = [c for c in edu_fields if c.startswith("last_screen.")]
has_last_value = edu_raw[last_screen_cols].apply(has_edu_value).any(axis=1)
n_with_value = has_last_value.sum()
n_null_end = (has_last_value & edu_raw["enrollments.end_date"].isna()).sum()
print(f"\nDQ: {n_with_value:,} rows have a last_screen education value; "
      f"{n_null_end:,} ({n_null_end / n_with_value * 100:.1f}%) have a null exit date")

# Pull custom KC assessment field with its own date — kept separate from enrollment fields
# to avoid a cross-product fan-out (assessments join differently than entry/last screens)
custom_fields = [
    "clients.personal_id",
    "client_assessments.added_date",
    "client_assessment_custom.c_kc_educational_history",
]

result_custom = run_query_with_retry(sdk, {
    "model": "wa_connection_model",
    "view": "base",
    "fields": custom_fields,
    "filters": {"enrollments.date_filter": "NULL"},
    "limit": "-1",
})
custom_raw = pd.read_csv(StringIO(result_custom), low_memory=False)

if len(custom_raw.columns) != len(custom_fields):
    raise ValueError(f"Custom column count mismatch: expected {len(custom_fields)}, got {len(custom_raw.columns)}")
custom_raw.columns = custom_fields
custom_raw["client_assessments.added_date"] = pd.to_datetime(custom_raw["client_assessments.added_date"], errors="coerce")
print(f"\n{len(custom_raw):,} rows for custom KC education field; "
      f"{custom_raw['clients.personal_id'].nunique():,} unique clients")

In [ ]:
# Two education output columns:
#   EducationCompleted: rhy_education_level, youth_education_recent_status, c_kc_educational_history
#   EducationStatus:    rhy_school_status, youth_education_enrollment, youth_education_current_status

today = pd.Timestamp.today()
pid = "clients.personal_id"
edu_raw["_entry_eff_date"] = edu_raw["enrollments.start_date"]
edu_raw["_last_eff_date"]  = edu_raw["enrollments.end_date"].fillna(today)

EXCLUDE_EXACT = {
    "", "nan", "99", "99.0", "8", "9",
    "client doesn't know", "client prefers not to answer", "data not collected",
}

def is_valid_edu(series):
    s_lower = series.astype(str).str.strip().str.lower()
    return series.notna() & ~s_lower.isin(EXCLUDE_EXACT)

def build_long(concepts, include_custom=False):
    parts = []
    for concept in concepts:
        for screen, date_col in [("entry_screen", "_entry_eff_date"), ("last_screen", "_last_eff_date")]:
            sub = edu_raw[[pid, f"{screen}.{concept}", date_col]].copy()
            sub.columns = [pid, "value", "eff_date"]
            sub["source_field"] = f"{screen}.{concept}"
            parts.append(sub)
    if include_custom:
        custom_col = "client_assessment_custom.c_kc_educational_history"
        sub = custom_raw[[pid, custom_col, "client_assessments.added_date"]].copy()
        sub.columns = [pid, "value", "eff_date"]
        sub["source_field"] = custom_col
        parts.append(sub)
    return pd.concat(parts, ignore_index=True)

def dedup_with_breakdown(long_df, out_col):
    filtered = long_df[is_valid_edu(long_df["value"])].copy()
    total = len(filtered)

    best = (
        filtered
        .sort_values("eff_date", ascending=False)
        .drop_duplicates(subset=pid, keep="first")
    )
    n_best = len(best)

    before = (
        filtered.groupby("source_field")["value"].count()
        .rename("n_values").reset_index()
    )
    before["pct_of_valid"] = (before["n_values"] / total * 100).round(1)

    after = (
        best.groupby("source_field")["value"].count()
        .rename("n_winning").reset_index()
    )
    after["pct_winning"] = (after["n_winning"] / n_best * 100).round(1)

    breakdown = (
        before.merge(after, on="source_field", how="left")
        .fillna(0)
        .assign(n_winning=lambda d: d["n_winning"].astype(int))
        .sort_values("n_values", ascending=False)
    )

    print(f"\n── {out_col}: {n_best:,} clients with a value ──")
    display(breakdown)

    return best[[pid, "value"]].rename(columns={pid: "PersonalID", "value": out_col})

completed_df = dedup_with_breakdown(
    build_long(["rhy_education_level", "youth_education_recent_status"], include_custom=True),
    "EducationCompleted",
)

status_df = dedup_with_breakdown(
    build_long(["rhy_school_status", "youth_education_enrollment", "youth_education_current_status"]),
    "EducationStatus",
)

edu_combined = completed_df.merge(status_df, on="PersonalID", how="outer")

# Unique value combinations with counts
print("\n── EducationCompleted × EducationStatus combinations ──")
combos = (
    edu_combined
    .groupby(["EducationCompleted", "EducationStatus"], dropna=False)
    .size()
    .reset_index(name="n_clients")
    .sort_values("n_clients", ascending=False)
    .reset_index(drop=True)
)
with pd.option_context("display.max_rows", 100):
    display(combos)

df0 = df0.merge(edu_combined, on="PersonalID", how="left")
print(f"\ndf0 after education join: {len(df0):,} rows")

In [ ]:
# ---------------------------------------------------------------------------
# Compute the new columns and merge onto df0
# ---------------------------------------------------------------------------
# Looker returns text values for disability fields (e.g. "Yes", "No",
# "Alcohol use disorder"). DQ responses are excluded by text match.
# income_cash is numeric.
# ---------------------------------------------------------------------------

# Standard HMIS DK/PNTA/DNC triad. Looker sometimes returns a curly apostrophe
# (’) in "doesn't know" — normalized to a straight one before comparing,
# so both spellings are caught with a single canonical entry.
DQ_TEXT_LOWER = {"client doesn't know", "client prefers not to answer", "data not collected"}

def _normalize_text(series):
    return series.astype(str).str.lower().str.replace("’", "'", regex=False)

def valid_mask(series):
    return series.notna() & ~_normalize_text(series).isin(DQ_TEXT_LOWER)

all_screens = (
    pd.concat([df.dropna(axis=1, how="all") for df in raw.values()], ignore_index=True)
    .assign(date=lambda df: pd.to_datetime(df["date"], format="mixed", errors="coerce"))
    .sort_values(["PersonalID", "date"])
)

all_screens["income_cash"] = pd.to_numeric(all_screens["income_cash"], errors="coerce")

# most_recent helper — last valid (non-null, non-DQ) text value per client
def most_recent_text(df, field, dq_lower=DQ_TEXT_LOWER):
    valid = df[df[field].notna() & ~_normalize_text(df[field]).isin(dq_lower)]
    return valid.groupby("PersonalID")[field].last()

# Substance use disorder — most recent response (text from Looker)
sub_recent = most_recent_text(all_screens, "health_substance_abuse").rename("SUDRecent")

# Mental health disorder — most recent response (text from Looker)
mh_recent = most_recent_text(all_screens, "health_mental").rename("MHDisorderRecent")

# Disabling condition — all-time ("Yes" if ever "Yes", else "No" if any valid data, else NaN)
dis_valid = all_screens[valid_mask(all_screens["disabling_condition"])]
dis_alltime = (
    dis_valid.groupby("PersonalID")["disabling_condition"]
    .apply(lambda x: "Yes" if (x == "Yes").any() else "No")
    .rename("DisablingConditionAnytime")
)

# Total monthly income — most recent screen with a recorded amount
inc_recent = (
    all_screens[all_screens["income_cash"].notna()]
    .groupby("PersonalID")["income_cash"]
    .last()
    .rename("MonthlyCashIncome")
)

# client_assessments and current_living_situation are already keyed by
# PersonalID (see pull above) — just align column names with all_screens.
assess_renamed = assess_raw.rename(columns={"AssessmentDate": "date"})

cls_renamed = cls_raw.rename(columns={
    "InformationDate": "date",
    "CurrentLivingSituation": "value",
})

# ---------------------------------------------------------------------------
# DV survivor status — combines "Survivor of DV" (health_dv, HMIS 4.11.2) with
# its conditional follow-up "Currently Fleeing" (health_dv_fleeing, 4.11.2B),
# which HUD only collects when health_dv = Yes. Both values are read from the
# SAME winning record (most recent valid health_dv response, across screens +
# client_assessments) rather than independently-most-recent per field, so the
# two answers can't be paired from two different, possibly contradictory,
# screening events.
# ---------------------------------------------------------------------------
dv_combined_source = pd.concat(
    [all_screens[["PersonalID", "date", "health_dv", "health_dv_fleeing"]],
     assess_renamed[["PersonalID", "date", "health_dv", "health_dv_fleeing"]]],
    ignore_index=True,
)
dv_winning = (
    dv_combined_source[valid_mask(dv_combined_source["health_dv"])]
    .sort_values(["PersonalID", "date"])
    .groupby("PersonalID")
    .tail(1)
    .set_index("PersonalID")
)

_health_dv_norm = _normalize_text(dv_winning["health_dv"])
_fleeing_valid = valid_mask(dv_winning["health_dv_fleeing"])
_fleeing_norm = _normalize_text(dv_winning["health_dv_fleeing"])

dv_status = pd.Series(
    np.select(
        [
            _health_dv_norm != "yes",
            _fleeing_valid & (_fleeing_norm == "yes"),
            _fleeing_valid & (_fleeing_norm == "no"),
        ],
        [
            "Not a Survivor",
            "Survivor - Currently Fleeing",
            "Survivor - Not Currently Fleeing",
        ],
        default="Survivor - Fleeing Status Unknown",  # health_dv=Yes but no valid fleeing response recorded
    ),
    index=dv_winning.index,
    name="DVSurvivorStatus",
)

# Sexual orientation — most recent value, screens + client_assessments combined
orientation_source = pd.concat(
    [all_screens[["PersonalID", "date", "rhy_sexual_orientation"]],
     assess_renamed[["PersonalID", "date", "rhy_sexual_orientation"]]],
    ignore_index=True,
).sort_values(["PersonalID", "date"])
orientation_recent = most_recent_text(orientation_source, "rhy_sexual_orientation").rename("SexualOrientationRecent")

# Translation services needed — entry_screen only, no client_assessments equivalent
translation_recent = most_recent_text(all_screens, "translation_assistance_needed").rename("TranslationAssistanceNeededRecent")

# ---------------------------------------------------------------------------
# Foster care / Justice / Behavioral health institution — each is Yes if EITHER
# a dedicated field says Yes (foster care, justice only) OR an institutional
# living-situation value was ever recorded (all three). Implemented as
# vectorized isin + groupby-any rather than a literal per-person loop, which
# is functionally equivalent to stopping the search on first confirmation.
# No vs. NaN: "No" means we have valid data somewhere and found no match;
# NaN means no valid data was ever found for that person (same convention as
# DisablingConditionAnytime above).
# ---------------------------------------------------------------------------

institutionalsituation_map = {
    "foster_care": ["Foster care home or foster care group home"],
    "justice": ["Jail, prison or juvenile detention facility"],
    "behavioral_health": [
        "Psychiatric hospital or other psychiatric facility",
        "Substance abuse treatment facility or detox center",
    ],
}

living_situation_long = pd.concat([
    all_screens[["PersonalID", "prior_residence_text"]].rename(columns={"prior_residence_text": "value"}),
    all_screens[["PersonalID", "exit_destination_text"]].rename(columns={"exit_destination_text": "value"}),
    assess_renamed[["PersonalID", "prior_residence_text"]].rename(columns={"prior_residence_text": "value"}),
    cls_renamed[["PersonalID", "value"]],
], ignore_index=True)
living_situation_long = living_situation_long[valid_mask(living_situation_long["value"])]
living_valid_ids = set(living_situation_long["PersonalID"])

def text_ids_matching(category):
    return set(living_situation_long.loc[living_situation_long["value"].isin(institutionalsituation_map[category]), "PersonalID"])

def field_positive_and_valid(sources_list, field):
    combined = pd.concat([df[["PersonalID", field]] for df in sources_list], ignore_index=True)
    valid = combined[valid_mask(combined[field])]
    positive = set(valid.loc[valid[field].str.lower() == "yes", "PersonalID"])
    return positive, set(valid["PersonalID"])

foster_field_positive, foster_field_valid = field_positive_and_valid([all_screens, assess_renamed], "previous_foster_care")
justice_field_positive, justice_field_valid = field_positive_and_valid([all_screens, assess_renamed], "rhy_former_justice")

foster_positive  = text_ids_matching("foster_care") | foster_field_positive
foster_valid     = living_valid_ids | foster_field_valid

justice_positive = text_ids_matching("justice") | justice_field_positive
justice_valid    = living_valid_ids | justice_field_valid

bh_positive = text_ids_matching("behavioral_health")
bh_valid    = living_valid_ids

def build_flag(positive_ids, valid_ids, name):
    idx = pd.Index(sorted(valid_ids), name="PersonalID")
    flag = pd.Series("No", index=idx, name=name)
    flag.loc[flag.index.isin(positive_ids)] = "Yes"
    return flag

foster_flag  = build_flag(foster_positive, foster_valid, "FosterCareInvolvement")
justice_flag = build_flag(justice_positive, justice_valid, "JusticeInvolvement")
bh_flag      = build_flag(bh_positive, bh_valid, "BehavioralHealthInstitution")

new_cols = (
    pd.concat([
        sub_recent, mh_recent, dis_alltime, inc_recent,
        dv_status, orientation_recent, translation_recent,
        foster_flag, justice_flag, bh_flag,
    ], axis=1)
    .reset_index()
)

df0 = df0.merge(new_cols, on="PersonalID", how="left")

for col in ["DVSurvivorStatus", "SexualOrientationRecent", "TranslationAssistanceNeededRecent",
            "FosterCareInvolvement", "JusticeInvolvement", "BehavioralHealthInstitution"]:
    print(f"\n{col}:")
    print(df0[col].value_counts(dropna=False))

col_order = [
    "ClientUniqueIdentifier", "PersonalID", "CurrentAge", "AgeTier", "FederallyRecognizedTribe",
    "LastKnownCity", "LastKnownSubregion",
    "Gender", "GenderExpanded",
    "Gender_Woman", "Gender_Man", "Gender_CSI", "Gender_Trans", "Gender_NB", "Gender_Q", "Gender_DI",
    "RaceAndEthnicity", "RaceAndEthnicityExpanded",
    "RaceAndEthnicity_AIAN", "RaceAndEthnicity_Asian", "RaceAndEthnicity_Black",
    "RaceAndEthnicity_NHPI", "RaceAndEthnicity_WC", "RaceAndEthnicity_HL", "RaceAndEthnicity_MENA",
    "VeteranStatus",
    "SUDRecent", "MHDisorderRecent", "DisablingConditionAnytime", "MonthlyCashIncome",
    "DVSurvivorStatus", "SexualOrientationRecent", "TranslationAssistanceNeededRecent",
    "FosterCareInvolvement", "JusticeInvolvement", "BehavioralHealthInstitution",
    "EducationCompleted", "EducationStatus",
    "DataAsOfDate",
]
df0 = df0[col_order]
print(f"\nMerged. df0 shape: {df0.shape}")
# df0[["PersonalID", "SUDRecent", "MHDisorderRecent",
#      "DisablingConditionAnytime", "MonthlyCashIncome"]].head()

In [ ]:
str_cols = [c for c in df0.columns if c not in ("CurrentAge", "MonthlyCashIncome", "DataAsOfDate")]
final = df0.copy()
final[str_cols] = final[str_cols].astype("string")
final["MonthlyCashIncome"] = pd.to_numeric(final["MonthlyCashIncome"], errors="coerce").astype("Float64")

In [ ]:
write_table(final, "Client_Demographics.parquet", dev=True)